# Visualizations

Plotly charts from categorized discovery records.
Run `notebook/analysis.ipynb` first so CSVs exist under `output/processed/`.

Paths are anchored at the project root (`simple/`), one level above this notebook.


In [33]:

%pip install -q -r {_requirements()}
%pip install -q "nbformat>=4.2.0"


zsh: parse error near `}'
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 0. Imports & paths


In [34]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import umap

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots


def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "scripts" / "sentiment_analysis.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find project root (expected scripts/sentiment_analysis.py). "
        f"cwd={here}"
    )


ROOT = _project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.categorize_records import EMBEDDING_MODEL

CATEGORY_ORDER = [
    "Workflow & Business Processes",
    "Case Management",
    "Data Management & Visibility",
    "Records & Document Management",
    "System Integration",
    "Reporting & Decision Support",
    "Communication & Collaboration",
    "Scheduling & Resource Management",
    "User Experience & Performance",
    "Training & Documentation",
]

PROCESSED = ROOT / "output" / "processed"
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CATEGORIZED_CHALLENGES_CSV = PROCESSED / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED / "categorized_expectations.csv"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"


## 1. Load categorized records

Reads `categorized_challenges.csv` / `categorized_expectations.csv` written by analysis.
The semantic map cell re-embeds challenges (embeddings are not persisted).


In [35]:
missing = [p for p in (CATEGORIZED_CHALLENGES_CSV, CATEGORIZED_EXPECTATIONS_CSV) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Run notebook/analysis.ipynb through the save step first. Missing:\n  - "
        + "\n  - ".join(str(p) for p in missing)
    )

df = pd.read_csv(CATEGORIZED_CHALLENGES_CSV)
expectations_df = pd.read_csv(CATEGORIZED_EXPECTATIONS_CSV)

print(f"Challenges: {len(df)} | Expectations: {len(expectations_df)}")
print(f"Categories: {df['Category'].nunique()} | Focus groups: {df['focus_group'].nunique()}")


Challenges: 1151 | Expectations: 540
Categories: 10 | Focus groups: 20


## 1b. Reconcile categories → `final_category`

Each record has its own hybrid `Category` (with `Category_Confidence`) and belongs
to a cluster whose dominant category is `Cluster_Label`. **Cluster purity** is the
share of a cluster whose per-record `Category` matches that dominant label.

`final_category` trusts the record's own category when it is confident, but adopts
the cluster's label when the record is low-confidence *and* sits in a pure cluster.

In [36]:
def add_final_category(frame, purity_threshold=0.6, low_conf=3.0):
    """Blend per-record Category with the cluster's dominant label (Cluster_Label).

    cluster purity = share of a cluster whose Category == Cluster_Label. When a
    record's own category is low-confidence (< low_conf) but it lives in a pure
    cluster (purity >= purity_threshold), adopt the cluster label; else keep Category.
    """
    frame = frame.copy()
    match = frame["Category"] == frame["Cluster_Label"]
    purity = match.groupby(frame["Cluster"]).transform("mean")
    frame["Cluster_Purity"] = np.where(frame["Cluster"] == -1, np.nan, purity)

    frame["final_category"] = frame["Category"]
    take_cluster = (
        (frame["Cluster"] != -1)
        & (frame["Cluster_Purity"] >= purity_threshold)
        & (frame["Category_Confidence"] < low_conf)
    )
    frame.loc[take_cluster, "final_category"] = frame.loc[take_cluster, "Cluster_Label"]
    frame["Category_Reassigned"] = frame["final_category"] != frame["Category"]
    return frame


df = add_final_category(df)
expectations_df = add_final_category(expectations_df)

for label, frame in [("Challenges", df), ("Expectations", expectations_df)]:
    n = int(frame["Category_Reassigned"].sum())
    print(f"{label}: {n}/{len(frame)} records took the cluster label "
          f"({100 * n / len(frame):.1f}%)")

Challenges: 17/1151 records took the cluster label (1.5%)
Expectations: 10/540 records took the cluster label (1.9%)


In [37]:
# Compare cluster purity against category confidence, per cluster (challenges).
purity_summary = (
    df[df["Cluster"] != -1]
    .groupby(["Cluster", "Cluster_Label"])
    .agg(
        records=("Category", "size"),
        purity=("Cluster_Purity", "first"),
        mean_confidence=("Category_Confidence", "mean"),
        reassigned=("Category_Reassigned", "sum"),
    )
    .reset_index()
    .sort_values("purity", ascending=False)
)
display(purity_summary.round(3))

fig_pc = px.scatter(
    df[df["Cluster"] != -1],
    x="Category_Confidence", y="Cluster_Purity",
    color="Cluster_Label",
    hover_data=["focus_group", "Category", "final_category"],
    title="Cluster purity vs category confidence (challenges)",
)
fig_pc.add_hline(y=0.6, line_dash="dash", line_color="#94A3B8", annotation_text="purity 0.6")
fig_pc.add_vline(x=3.0, line_dash="dash", line_color="#94A3B8", annotation_text="confidence 3.0")
fig_pc.update_layout(height=520, width=1000, template="plotly_white")
fig_pc.show()

,Cluster,Cluster_Label,records,purity,mean_confidence,reassigned
15,15,Scheduling & Resource Management,12,1.000,9.730,0
17,17,Case Management,20,0.850,3.951,2
18,18,Case Management,18,0.833,5.439,0
4,4,Case Management,56,0.750,5.715,5
2,2,User Experience & Performance,18,0.667,5.376,1
7,7,User Experience & Performance,14,0.643,3.632,1
0,0,System Integration,16,0.625,3.069,3
16,16,Case Management,20,0.600,4.217,5
8,8,User Experience & Performance,52,0.596,4.696,0
21,21,Records & Document Management,52,0.538,4.479,0


## 2. Category counts


In [38]:
category_counts = (
    df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_cat = px.bar(
    category_counts, x="Count", y="Category", orientation="h",
    title="Pain point categories", color="Count", color_continuous_scale="Blues",
)
fig_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_cat.show()

expectation_counts = (
    expectations_df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_exp_cat = px.bar(
    expectation_counts, x="Count", y="Category", orientation="h",
    title="Expectation categories", color="Count", color_continuous_scale="Teal",
)
fig_exp_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_exp_cat.show()


## 4. Challenges vs expectations by theme


In [39]:
ch_c = df["Category"].value_counts()
ex_c = expectations_df["Category"].value_counts()
cats = sorted(set(ch_c.index) | set(ex_c.index), key=lambda c: -(ch_c.get(c, 0) + ex_c.get(c, 0)))
plot_cmp = pd.DataFrame({
    "Category": cats,
    "Challenges": [int(ch_c.get(c, 0)) for c in cats],
    "Expectations": [int(ex_c.get(c, 0)) for c in cats],
}).melt(id_vars="Category", var_name="Type", value_name="Count")
fig_cmp = px.bar(
    plot_cmp, x="Count", y="Category", color="Type", barmode="group", orientation="h",
    title="Challenges vs expectations by theme",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
)
fig_cmp.update_layout(height=520, template="plotly_white", yaxis=dict(categoryorder="total ascending"))
fig_cmp.show()


## 5. Top focus groups


In [40]:
top_n = 20
top = df.groupby("focus_group").size().sort_values(ascending=False).head(top_n)
exp = expectations_df.groupby("focus_group").size()
plot_top = pd.DataFrame({
    "Focus Group": top.index.tolist(),
    "Challenges": top.values.astype(int),
    "Expectations": [int(exp.get(g, 0)) for g in top.index],
}).melt(id_vars="Focus Group", var_name="Kind", value_name="Count")
fig_top = px.bar(
    plot_top,
    x="Focus Group",
    y="Count",
    color="Kind",
    barmode="group",
    title=f"Top {top_n} focus groups",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
    category_orders={"Focus Group": top.index.tolist()},
)
fig_top.update_layout(
    height=560,
    width=1000,
    template="plotly_white",
    xaxis=dict(tickangle=-40),
    margin=dict(b=140),
)
fig_top.show()


## 6. Category heatmaps

- **Dense cluster × category** — each cluster is its own row (id + dominant label),
  columns are the per-record `Category`; diagonal-heavy rows are pure clusters.
- **Interactive focus group × final category** — click-free drill-down further down.

In [41]:
# Dense heatmap: each cluster (row) × per-record Category (column).
# Two labels per record (Cluster_Label + Category) spread challenges across more
# cells, so the grid is denser than a focus-group × category view.
cc = df[df["Cluster"] != -1].copy()
cc["Cluster · label"] = "C" + cc["Cluster"].astype(str) + " · " + cc["Cluster_Label"]
heatmap_df = pd.crosstab(cc["Cluster · label"], cc["Category"])
heatmap_df = heatmap_df.reindex(columns=[c for c in CATEGORY_ORDER if c in heatmap_df.columns])
heatmap_df = heatmap_df.loc[heatmap_df.sum(axis=1).sort_values(ascending=False).index]
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Category", y="Cluster · dominant label", color="Count"),
    color_continuous_scale="YlOrRd", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(
    title="Challenges: cluster × category (dense)",
    height=max(600, 24 * len(heatmap_df) + 140), width=1050,
    xaxis_tickangle=-45, template="plotly_white",
)
fig_heatmap.show()

In [42]:
# Dense heatmap: each cluster (row) × per-record Category (column).
# Two labels per record (Cluster_Label + Category) spread expectations across more
# cells, so the grid is denser than a focus-group × category view.
cc = expectations_df[expectations_df["Cluster"] != -1].copy()
cc["Cluster · label"] = "C" + cc["Cluster"].astype(str) + " · " + cc["Cluster_Label"]
heatmap_df = pd.crosstab(cc["Cluster · label"], cc["Category"])
heatmap_df = heatmap_df.reindex(columns=[c for c in CATEGORY_ORDER if c in heatmap_df.columns])
heatmap_df = heatmap_df.loc[heatmap_df.sum(axis=1).sort_values(ascending=False).index]
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Category", y="Cluster · dominant label", color="Count"),
    color_continuous_scale="Teal", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(
    title="Expectations: cluster × category (dense)",
    height=max(600, 24 * len(heatmap_df) + 140), width=1050,
    xaxis_tickangle=-45, template="plotly_white",
)
fig_heatmap.show()

### Department × final category

Same heatmap grouped by department instead of focus group.

In [43]:
heatmap_df = pd.crosstab(df["department"], df["final_category"])
department_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[department_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Department", y="Final category", color="Count"),
    color_continuous_scale="YlOrRd", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(
    title="Challenges: department × final category",
    height=700, width=1000, xaxis_tickangle=-45, template="plotly_white",
)
fig_heatmap.show()

In [44]:
heatmap_df = pd.crosstab(expectations_df["department"], expectations_df["final_category"])
department_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[department_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Department", y="Final category", color="Count"),
    color_continuous_scale="Teal", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(
    title="Expectations: department × final category",
    height=700, width=1000, xaxis_tickangle=-45, template="plotly_white",
)
fig_heatmap.show()

In [ ]:
# Truly clickable heatmap for the Cursor / VS Code notebook renderer.
# plotly FigureWidget.on_click does NOT fire here, but ipywidgets Button.on_click does,
# so the heatmap is drawn as a grid of colored buttons — click a cell to list its records.
import matplotlib
from ipywidgets import Button, GridBox, Layout, Output, HTML, VBox
from IPython.display import display, clear_output


def clickable_heatmap(frame, text_col, noun, cmap_name, out_html, category_col="final_category"):
    heat = pd.crosstab(frame["focus_group"], frame[category_col])
    heat = heat.reindex(columns=[c for c in CATEGORY_ORDER if c in heat.columns])
    heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index]
    focus_groups = heat.index.tolist()   # grid columns
    categories = heat.columns.tolist()   # grid rows
    vmax = max(1, int(heat.values.max()))
    cmap = matplotlib.colormaps[cmap_name]

    def cell_color(v):
        r, g, b, _ = cmap(v / vmax)
        return f"rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})"

    # Static Plotly copy for export/sharing (same orientation: categories on Y).
    zt = heat.T
    go.Figure(go.Heatmap(
        z=zt.values, x=focus_groups, y=categories, colorscale=cmap_name,
        text=zt.values, texttemplate="%{text}", xgap=1, ygap=1,
    )).update_layout(
        title=f"{noun} volume: focus group × final category",
        template="plotly_white", height=600, width=1100,
        xaxis=dict(tickangle=-40), yaxis=dict(autorange="reversed"),
    ).write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved {out_html}")

    detail_cols = [
        c for c in ["focus_group", text_col, "final_category", "Category",
                    "Category_Confidence", "Cluster_Label", "Cluster_Purity"]
        if c in frame.columns
    ]
    out = Output()

    def show_records(fg, cat):
        sub = frame[(frame["focus_group"] == fg) & (frame[category_col] == cat)][detail_cols].drop_duplicates()
        if "Category_Confidence" in sub.columns:
            sub = sub.sort_values("Category_Confidence", ascending=False)
        with out:
            clear_output(wait=True)
            print(f"{len(sub)} {noun.lower()} records · {fg} × {cat}")
            display(sub if not sub.empty else "No records for this selection.")

    col_w, label_w, header_h = 48, 230, 150
    cells = [HTML("")]  # top-left corner
    for fg in focus_groups:  # rotated column headers
        cells.append(HTML(
            f"<div title='{fg}' style='height:{header_h}px;display:flex;align-items:flex-end;"
            f"justify-content:center'><span style='writing-mode:vertical-rl;transform:rotate(180deg);"
            f"font-size:11px;font-weight:600;white-space:nowrap'>{fg}</span></div>"))
    for cat in categories:
        cells.append(HTML(
            f"<div style='height:34px;display:flex;align-items:center;justify-content:flex-end;"
            f"padding-right:6px;font-size:11px;font-weight:600'>{cat}</div>"))
        for fg in focus_groups:
            v = int(heat.loc[fg, cat])
            b = Button(description=(str(v) if v else ""),
                       tooltip=f"{cat} × {fg}: {v} — click to view",
                       layout=Layout(width=f"{col_w}px", height="34px", margin="0px"))
            b.style.button_color = cell_color(v)
            if v / vmax >= 0.55:
                b.style.text_color = "white"
            b.on_click(lambda _b, fg=fg, cat=cat: show_records(fg, cat))
            cells.append(b)

    grid = GridBox(cells, layout=Layout(
        grid_template_columns=f"{label_w}px " + f"{col_w}px " * len(focus_groups),
        grid_gap="1px"))
    display(VBox([
        HTML(f"<h4 style='margin:4px 0'>{noun} volume: focus group × final category "
             f"<span style='font-weight:400;color:#64748b'>(click a cell to view records)</span></h4>"),
        grid, out]))


# Challenges
clickable_heatmap(
    df, CHALLENGE_TEXT_COL, "Challenge", "YlOrRd",
    FIG_DIR / "heatmap_focus_category.html",
)

In [ ]:
# Expectations (same clickable heatmap)
clickable_heatmap(
    expectations_df, EXPECTATION_TEXT_COL, "Expectation", "GnBu",
    FIG_DIR / "heatmap_focus_category_expectations.html",
)